<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🤖 Lab 02: Create Your First Prompted Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we create our first **Prompted Agent** using the Azure AI Projects SDK — the **Contoso Travel Concierge**. We'll learn how to define an agent with custom instructions, start conversations, send messages, and handle multi-turn interactions. The concierge is the front-desk agent that greets customers and helps answer general travel questions. In later labs, we'll enhance it with specialized tools and connect it to domain-specific agents.

---

## 🧳 The Contoso Travel Story

**Contoso Travel** needs an AI concierge — the friendly front-desk agent that greets every customer, understands what kind of trip they're looking for, and guides them through the planning process. Think of the best travel advisor you've ever worked with: warm, knowledgeable, and able to hold a natural conversation about your dream vacation. **You've got your environment set up** (Labs 00–01). The SDK is installed, credentials are validated, and the sample data is loaded. Now it's time to build the actual agent.

### 🔴 The Problem You Face Right Now

- You've never built an AI agent with the Foundry SDK before.
- You know you need to give it a personality, define its behavior through instructions, and interact with it through conversations — but **how do you actually create one?**
- What's the right way to define instructions? How do conversations work? How do you handle multi-turn interactions where the customer refines their request?
- And how do you clean up resources when you're done?

### ✅ What This Lab Solves

This lab takes you from zero to a working **Prompted Agent** — the Contoso Travel Concierge:
 - defining an agent with `PromptAgentDefinition`,
 - creating conversations and sending messages,
 - handling multi-turn dialogue, and
 - inspecting response objects.

This is the foundation that every other lab builds on.

### 🧠 Mental Model: Agents as Team Members

Think of building an AI agent like hiring a new team member for Contoso Travel:

| Real World | Agent Concept | SDK Component |
|------------|---------------|---------------|
| Write a job description | Define agent instructions | `PromptAgentDefinition` |
| Assign them a role | Give the agent a name | `agent_name` parameter |
| Start a meeting | Open a conversation | `conversations.create()` |
| Ask a question | Send a message | `responses.create()` |
| Continue discussing | Multi-turn follow-up | Same `conversation.id` |
| End the meeting | Delete conversation | `conversations.delete()` |

The **Concierge** is your first hire — a friendly front-desk agent. In later labs, you'll hire specialists (flights, hotels, cars) and a manager (orchestrator) to coordinate them.

## 1. Setup

Load environment variables and create the Azure AI Project client.

---


In [ ]:
# Setup: load credentials and create SDK clients
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
# PromptAgentDefinition: declarative config for an
# LLM agent (model + instructions + optional tools)
from azure.ai.projects.models import PromptAgentDefinition

# .env lives one level above the notebooks dir
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
# AIProjectClient manages agents; openai_client handles
# conversations and responses via OpenAI-compatible API
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")

## 2. Create the Concierge Agent

We define the agent with travel-focused instructions. The system prompt shapes how the agent responds to customer queries.

---


In [ ]:
# Define the system prompt — shapes the agent's
# personality, boundaries, and response style
CONCIERGE_INSTRUCTIONS = """You are the Contoso Travel Concierge, a friendly and knowledgeable travel assistant.

Your responsibilities:
- Help customers plan trips by answering questions about destinations, travel tips, and logistics
- Provide helpful, accurate, and concise travel advice
- Be warm and professional in your responses
- When you don't have specific data, provide general travel guidance
- Always mention that Contoso Travel can help with flights, hotels, and car rentals

Remember: You are representing Contoso Travel, a premium travel agency. 
Keep responses focused and helpful."""

# create_version creates an immutable snapshot of the
# agent. Each call yields a new version — useful for
# A/B testing, rollback, and audit trails.
agent = project_client.agents.create_version(
    agent_name="contoso-travel-concierge",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=CONCIERGE_INSTRUCTIONS,
    ),
)

print(f"✅ Agent created!")
print(f"   Name: {agent.name}")
print(f"   Version: {agent.version}")
print(f"   ID: {agent.id}")

## 3. Start a Conversation

Conversations maintain context across multiple messages. Let's create one and send our first travel query.

---


In [ ]:
# Create a conversation — a server-side session that
# tracks message history for multi-turn context
conversation = openai_client.conversations.create()
print(f"✅ Conversation created (ID: {conversation.id})")

# agent_reference in extra_body binds this response
# to a specific named agent (uses latest version)
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Hi! I'm thinking about planning a trip to Paris. What should I know?",
)

# output_text: convenience accessor that concatenates
# all text content from response.output items
print(f"\n🤖 Concierge Response:\n")
print(response.output_text)

## 4. Multi-Turn Conversation

The agent maintains context within a conversation. Let's ask follow-up questions to see how it handles multi-turn dialogue.

---


In [ ]:
# Same conversation.id = agent retains full context.
# It knows we're asking about Paris without restating.
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="What's the best time of year to visit?",
)

print(f"🤖 Concierge Response:\n")
print(response.output_text)

In [ ]:
# Third turn — agent still has full Paris conversation
# history from the server-side message store
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Can you suggest what I should budget for a week-long trip?",
)

print(f"🤖 Concierge Response:\n")
print(response.output_text)

## 5. Explore the Response Object

Let's look at the structure of the response to understand what data is available.

---


In [ ]:
# Inspect the response object structure.
# output is a list of typed items (e.g., 'message'
# items containing content blocks like text).
print(f"Response ID:     {response.id}")
print(f"Model:           {response.model}")
print(f"Status:          {response.status}")
print(f"Output Text:     {response.output_text[:100]}...")
print(f"\nOutput Items ({len(response.output)} items):")
for i, item in enumerate(response.output):
    print(f"  [{i}] Type: {item.type}")
    if hasattr(item, 'content'):
        for j, content in enumerate(item.content):
            print(f"       Content[{j}]: {type(content).__name__}")

# Token usage — track this for cost monitoring
print(f"\nUsage:")
print(f"  Input tokens:  {response.usage.input_tokens}")
print(f"  Output tokens: {response.usage.output_tokens}")
print(f"  Total tokens:  {response.usage.total_tokens}")

## 6. Cleanup

Always clean up resources when done — delete the conversation and agent version.

---


In [ ]:
# Cleanup: free server-side resources.
# Conversations persist until explicitly deleted.
openai_client.conversations.delete(conversation_id=conversation.id)
print("✅ Conversation deleted")

# Deletes this version only; other versions (if any)
# of the same agent name remain available.
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent version deleted")

# Release HTTP connections and token caches
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 7. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Created a Prompted Agent with custom travel concierge instructions</li>
  <li>Demonstrated multi-turn context retention</li>
  <li>Explored the response object structure</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 Key Takeaway:</strong> The Contoso Travel concierge can answer general questions, but it doesn't have access to actual flight, hotel, or car rental data. It relies on its training data for responses.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> In <a href="lab-03a-tools.ipynb">Lab 03a</a>, we'll add <b>Function Tools</b> to give the agent access to our Contoso Travel CSV data — making it a truly data-driven travel assistant!
</div>

---
